In [3]:
import soundfile as sf
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CACA = ROOT / "data" / "raw" / "audio_src_caca" / "raw_finetune"
SEG_CACA = ROOT / "data" / "raw" / "audio_src_segments_caca" / "segments"
CSV_CACA = ROOT / "data" / "output" / "transkrip_acuan_caca.csv" 

YT = {
    "pkk1":"19:41","pkk2":"16:52","pkk3":"10:44","pkk4":"10:28","pkk5":"23:28",
    "rpl1":"17:28","rpl2":"21:50","rpl3":"24:15","rpl4":"27:38","rpl5":"24:04",
    "rpl6":"18:17","rpl7":"20:14","rpl8":"11:08","rpl9":"14:40","rpl10":"16:48",
    "rpl11":"28:30",
    "jarnir1":"26:03","jarnir2":"21:56","jarnir3":"26:56","jarnir4":"30:56","jarnir5":"14:42",
}
def mmss(t):
    m, d = t.split(":"); return int(m)*60 + int(d)

print("WAV mentah di:", RAW_CACA, "\n")

csv = pd.read_csv(CSV_CACA, dtype={"duration_sec": float}, keep_default_na=False)
seg = csv.groupby("source_video")["duration_sec"].agg(n_baris="count", seg_detik="sum")

baris = []
for w in sorted(RAW_CACA.glob("*.wav")) + sorted(RAW_CACA.glob("*.WAV")):
    i = sf.info(str(w))
    v = w.stem
    n_wav = len(list(SEG_CACA.glob(f"{v}_seg*.wav"))) if SEG_CACA.exists() else None
    baris.append({
        "video": v,
        "wav_detik": round(i.duration, 1),
        "yt_detik": mmss(YT[v]) if v in YT else None,
        "sr": i.samplerate, "ch": i.channels,
        "n_baris_csv": int(seg.loc[v, "n_baris"]) if v in seg.index else 0,
        "n_wav_segmen": n_wav,
        "seg_detik": round(seg.loc[v, "seg_detik"], 1) if v in seg.index else 0,
    })

df = pd.DataFrame(baris)
df["selisih_yt"] = (df.wav_detik - df.yt_detik).round(1)
df["cakupan"] = (df.seg_detik / df.wav_detik).round(3)
print(df.to_string(index=False))

print("\n--- Pemeriksaan ---")
print("Laju cuplik seragam :", df.sr.nunique() == 1, f"({sorted(df.sr.unique())})")
print("Kanal seragam       :", df.ch.nunique() == 1, f"({sorted(df.ch.unique())})")
beda = df[df.selisih_yt.abs() > 2]
print("WAV ≠ durasi YouTube:", "tidak ada" if beda.empty else f"\n{beda[['video','wav_detik','yt_detik','selisih_yt']].to_string(index=False)}")
mismatch = df[df.n_baris_csv != df.n_wav_segmen]
print("Jumlah segmen ≠ CSV :", "tidak ada" if mismatch.empty else f"\n{mismatch[['video','n_baris_csv','n_wav_segmen']].to_string(index=False)}")

WAV mentah di: c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\raw\audio_src_caca\raw_finetune 

  video  wav_detik  yt_detik    sr  ch  n_baris_csv  n_wav_segmen  seg_detik  selisih_yt  cakupan
jarnir1     1140.0      1563 48000   2          123           123      926.6      -423.0    0.813
jarnir3     1090.0      1616 48000   2          106           106     1002.2      -526.0    0.919
jarnir4     1140.0      1856 48000   2          103           103     1056.6      -716.0    0.927
jarnir5      882.0       882 48000   2           50            50      853.3         0.0    0.967
   pkk1     1181.0      1181 48000   2          127           127      746.0         0.0    0.632
   pkk2     1012.0      1012 48000   2          116           116      809.4         0.0    0.800
   pkk3      644.0       644 48000   2           62            62      565.4         0.0    0.878
   pkk5     1146.0      1408 48000   2          119           119     1047.2 

In [8]:
# pip install silero-vad torch librosa soundfile
import numpy as np, soundfile as sf, librosa, torch, pandas as pd
from pathlib import Path
from silero_vad import load_silero_vad, get_speech_timestamps

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CACA = ROOT / "data" / "raw" / "audio_src_caca" / "raw_finetune"
CSV_CACA = ROOT / "data" / "output" / "transkrip_acuan_caca.csv"   # sesuaikan

SR, MIN_SEG, MAX_SEG = 16000, 3.0, 20.0
VID = "jarnir1"

# --- persis seperti skrip Caca ---
x, sr = sf.read(str(RAW_CACA / f"{VID}.wav"))
if x.ndim > 1: x = np.mean(x, axis=1)
if sr != SR:   x = librosa.resample(x, orig_sr=sr, target_sr=SR)
peak = np.max(np.abs(x))
if peak > 0:   x = x / peak * 0.95

ts = get_speech_timestamps(torch.from_numpy(x).float(), load_silero_vad(),
                           sampling_rate=SR, min_speech_duration_ms=1000,
                           speech_pad_ms=300, min_silence_duration_ms=500)

durasi = []
for t in ts:
    a, b = t["start"], t["end"]
    potong = []
    if (b - a) / SR > MAX_SEG:
        pos = a
        while pos < b:
            e = min(pos + int(MAX_SEG * SR), b); potong.append((pos, e)); pos = e
    else:
        potong = [(a, b)]
    for p, q in potong:
        d = (q - p) / SR
        if d >= MIN_SEG: durasi.append(round(d, 2))

# --- bandingkan ---
csv = pd.read_csv(CSV_CACA, keep_default_na=False)
asli = csv.loc[csv.source_video == VID, "duration_sec"].round(2).tolist()

print(f"hasil rekonstruksi : {len(durasi)} segmen, {sum(durasi):.1f} dtk")
print(f"CSV Caca           : {len(asli)} segmen, {sum(asli):.1f} dtk")
if durasi == asli:
    print("\nCOCOK PERSIS — berkas 1140 detik itu memang masukan Caca.")
else:
    n = sum(1 for a, b in zip(durasi, asli) if a == b)
    print(f"\nTidak identik. {n} segmen pertama cocok dari {min(len(durasi), len(asli))}.")

hasil rekonstruksi : 123 segmen, 926.6 dtk
CSV Caca           : 123 segmen, 926.6 dtk

COCOK PERSIS — berkas 1140 detik itu memang masukan Caca.
